<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_31_archive_replay_case_study.ipynb">9.31 公开归档数据复盘</a></li>
        <li>下一节： <a href="9_33_lightweight_sample_dataset_package.ipynb">9.33 轻量样本数据集与练习包</a></li>
    </ul>
</div>


## 9.32 文件级示例仓库与发布包：从证据链到可复查结果

第 9.31 节把公开归档数据复盘写成一条证据链：科学问题决定检索条件，metadata 与 weblog 决定数据是否进入下一步，最小重跑使处理参数匹配目标量，产品差异审计决定结论边界。还差最后一层：这些证据怎样落到文件，使外部复查不依赖口头记忆、个人路径或临时 notebook 状态。

射电干涉测量的处理结果很少只由一个图像决定。一个结果可以写成

$$
R = F(D, C, \theta, E, M),
$$

其中 $D$ 表示输入数据身份，$C$ 表示校准和 flag 状态，$\theta$ 表示成像、源搜索和测量参数，$E$ 表示软件环境，$M$ 表示人工判断，如 mask、排除区域、异常说明和质量判读。文件级发布包的任务，就是让 $D,C,\theta,E,M$ 都能被识别，并把它们和最终结果 $R$ 绑定起来。

这里的“发布包”不一定意味着正式论文数据发布。教学项目、小组复盘、课程作业、内部 QA 和公开论文都可以使用同一套思想，只是规模不同。小型案例可以只有少量清单、参数和报告；正式研究需要更完整的产品身份、许可证、版本标签和长期归档。关键不是形式复杂，而是避免一个常见失败：图像和源表留下了，生成它们的数据版本、参数和人工判断却消失了。


![文件级发布包结构](figures/practical_file_release_package_map.png)

图 9.32.1 展示了文件级发布包的基本思想。数据清单、环境记录、参数文件、命令日志、原产品身份、重跑产品、差异审计和限制声明共同围绕同一个发布包。发布包不是把大型数据文件全部塞进版本库，而是让每个结果都有可追溯身份。


### 9.32.1 方法可以描述，结果必须落到文件

论文或报告中的方法段通常会写“使用某软件完成校准和成像，采用某种权重和阈值”。这种描述能说明大方向，却不足以复查一个具体结果。射电图像对参数、mask、flag、权重和主波束区域非常敏感。两个处理流程看似使用相同软件和相同权重，也可能因为不同 flag 版本、不同自动 mask 初始条件、不同通道合并方式或不同软件小版本而产生可见差异。

文件级示例仓库解决的是“具体性”问题。它把方法段中的抽象描述拆成可检查对象：输入数据清单、校准表清单、参数文件、运行日志、产品清单、验证报告和限制声明。这样一来，复查不再从猜测命令开始，而是从固定身份开始。若某个结果发生变化，可以追踪变化来自输入、环境、参数、人工判断还是验证标准。

这种设计尤其适合教学。教材正文可以解释原理，示例仓库可以展示同一原理在文件层面怎样落地。例如，正文讨论自然权重和 taper 如何改善低面亮度灵敏度；仓库则保留 `imaging_low_surface_brightness.yaml`、重跑日志、taper 图像、共同 beam 差异图和上限计算报告。理论、参数和产品因此能够相互指认。


### 9.32.2 发布包的最小组成

一个最小但可复查的发布包应包含七类对象。第一类是数据身份，记录归档项目、观测日期、执行块、field、spectral window、文件大小和校验和。第二类是处理身份，记录校准表、flag 状态、pipeline 版本、人工修改和运行编号。第三类是参数身份，记录权重、taper、cell size、image size、clean 阈值、mask、源搜索阈值和测量区域。第四类是环境身份，记录软件版本、容器或环境文件、关键依赖和操作系统信息。第五类是产品身份，记录图像、残差、模型、源表、region、通量表和报告的路径、单位、beam、噪声和校验和。第六类是验证结论，记录哪些门限通过、哪些失败、哪些例外被接受。第七类是限制声明，说明结果适用的频率、角尺度、主波束范围、信噪比范围和系统误差。

目录结构可以保持朴素。重点不是让文件夹看起来复杂，而是让对象之间的关系稳定。

```text
replay_release_package/
  README.md
  manifests/
    data_manifest.yaml
    calibration_manifest.yaml
    product_manifest.yaml
  configs/
    imaging.yaml
    source_finding.yaml
    measurement_regions.reg
  environment/
    software_versions.txt
    container_or_environment.yaml
  logs/
    command_log.txt
    manual_decisions.md
  products/
    images/
    residuals/
    catalogs/
    measurements/
  validation/
    qa_summary.md
    product_comparison.md
    limit_statement.md
```

这个结构中，`README.md` 负责说明包的目的和入口，清单文件负责身份，参数文件负责处理选择，日志负责运行过程，产品目录负责结果，验证目录负责判断。大型原始数据可以只通过清单引用；小型教学数据可以作为精简样本进入仓库。二者并不冲突。


### 9.32.3 清单文件：大文件不入库，身份必须入库

大型 Measurement Set、ASDM 和 FITS cube 通常不适合直接放进版本库。它们体积大，更新频繁，也可能受访问权限限制。但这并不意味着数据身份可以省略。没有数据身份，后续所有结果都处于悬空状态：同名文件可能来自不同下载批次，归档产品可能被重新发布，局部文件可能损坏，flag 版本可能不一致。

清单文件是一份契约，定义什么可以被复查。它不只是列出文件名，还要记录“怎样确认这是同一份输入”。一个数据清单可以包含如下字段：

```yaml
run_id: replay_lband_halo_2026_05_04
science_target: low_surface_brightness_continuum
inputs:
  - role: calibrated_measurement_set
    archive_project: example_archive_project
    execution_block: example_eb_01
    field: target_field
    spectral_windows: [0, 1, 2, 3]
    downloaded_on: 2026-05-04
    size_bytes: 18423391232
    checksum: sha256:example_checksum
    quality_report: qa/weblog_summary.md
```

参数清单同样需要固定科学选择。对于低面亮度连续谱，权重、taper、mask、主波束裁剪、源扣除策略和上限区域都属于科学判断，而不是纯粹的软件细节。

```yaml
imaging:
  weighting: natural
  uv_taper_arcsec: 30
  multiscale_scales_pixels: [0, 5, 15, 45]
  primary_beam_minimum: 0.30
  restored_beam_arcsec: [35.0, 32.0]
measurement:
  region_file: configs/halo_region.reg
  report_units: [Jy, K]
  include_flux_scale_error: true
```

清单的价值在于比较边界。若两次运行的 `checksum` 相同、环境相同、参数不同，则差异主要来自处理选择；若输入校验不同，则不能直接把产品差异解释为参数效果；若参数相同但软件版本不同，差异审计需要纳入环境变化。这种边界判断比单纯保留命令更重要。


![清单文件契约](figures/practical_file_manifest_contract.png)

图 9.32.2 把清单文件拆成数据、环境、参数、产品和验证五类对象。每一类都服务于一个复查问题：输入是否相同，运行是否可识别，科学目标是否一致，结果是否可比较，是否进入发布。


### 9.32.4 溯源图与陈旧产品

一个发布包中最容易出错的地方，是旧产品与新参数混在一起。成像参数改了，但源表没有重新生成；mask 改了，但通量表仍来自旧图像；flag 版本改了，但残差图仍被放进最终报告。这些错误在文件名相似时很隐蔽，尤其是多个小组成员并行处理同一数据时。

溯源图用于说明每个产品由哪些输入和判断共同生成。它可以很简单，不一定需要复杂工作流系统。关键是把边写清楚：图像依赖数据、校准表、成像参数和环境；源表依赖图像、残差、背景估计和源搜索参数；通量表依赖图像、beam、region、主波束裁剪和误差预算。任一上游对象改变，下游产品就必须被标记为陈旧或重新生成。


![产品溯源图](figures/practical_file_provenance_graph.png)

图 9.32.3 中，归档数据、校准表、参数文件和人工记录共同进入成像、源搜索和测量运行，再生成图像、源表和报告。运行编号把全部边绑定到一起。若任一输入或人工判断改变，相关产品不能继续被当作当前结果。


在小型项目中，溯源关系可以写进 `product_manifest.yaml`：

```yaml
products:
  - path: products/images/target_tapered.image.fits
    kind: restored_image
    run_id: replay_lband_halo_2026_05_04
    depends_on:
      - manifests/data_manifest.yaml
      - manifests/calibration_manifest.yaml
      - configs/imaging.yaml
      - environment/software_versions.txt
    beam_arcsec: [35.0, 32.0, -12.0]
    rms_jy_per_beam: 0.000045
    checksum: sha256:example_image_checksum
  - path: products/catalogs/source_catalog.ecsv
    kind: continuum_catalog
    depends_on:
      - products/images/target_tapered.image.fits
      - configs/source_finding.yaml
      - logs/manual_decisions.md
    checksum: sha256:example_catalog_checksum
```

这个清单让“陈旧产品”可以被机械识别。若 `configs/imaging.yaml` 改变，而图像产品的运行编号仍旧，处理报告必须说明图像尚未更新；若图像更新而源表仍依赖旧图像，源表不能进入发布。即使没有自动化系统，这种显式依赖也能显著降低混乱。


### 9.32.5 验证门：从候选产品到可发布结论

发布包中的产品不应默认都是最终结果。更稳健的做法是设置验证门，让产品从候选状态逐级进入可发布状态。输入校验失败时，处理停止；环境不可识别时，运行只能作为临时尝试；参数与科学目标不一致时，产品只能作为对照；校准证据不足时，图像只能形成候选结论；成像质量不达标时，源表和通量测量不能进入发布；科学测量的误差预算不完整时，报告只能保留为内部记录。

验证门的好处是把失败也变成材料。真实训练中，失败样本往往比顺利样本更有价值。一个残差结构明显的图像可以训练旁瓣和校准误差识别；一个主波束边缘伪检很多的源表可以训练选择函数判断；一个最大可恢复尺度不足的非探测可以训练限制声明。关键是失败原因必须留下，而不是把失败产品删除。


![验证门与失败样本](figures/practical_file_validation_gates.png)

图 9.32.4 展示了输入校验、环境校验、参数一致、校准证据、成像质量和科学测量六个验证门。产品只有通过相应门限后才进入发布结论；失败产品保留原因，用于误差判断和重跑边界训练。


验证报告不需要冗长，但应足够具体。例如，成像质量门可以记录 beam、rms、残差峰值、残差结构、主波束范围、强源附近排除区域和预期热噪声比值。源表质量门可以记录检测阈值、局部噪声估计、边缘区域排除、人工检查样本、完整性注入和可靠性估计。通量测量门可以记录 region、相关噪声修正、通量标尺误差、主波束误差和背景选择。

一个简洁的验证结论可以写成如下格式：

```yaml
validation:
  input_checksum: pass
  environment_identity: pass
  calibration_evidence: pass_with_note
  imaging_quality: pass_with_exclusion_region
  catalog_reliability: not_applicable
  science_measurement: pass_limited
release_level: partial_use
limit_statement: validation/limit_statement.md
notes:
  - strong_source_residual_excluded_from_measurement_region
  - maximum_recoverable_scale_limits_large_halo_interpretation
```

这里的 `partial_use` 不是缺陷，而是结论等级。它说明数据能支持某些量，不能支持另一些量。这样的发布包比只给出一张最终图更接近研究实践。


### 9.32.6 教学版本：小样本、标准包与完整包

文件级示例仓库可以按规模分层。小样本版本使用裁剪后的图像、简化清单和少量报告，适合演示清单、参数和产品比较。标准版本包含归档产品、重跑参数、差异图、源表和验证报告，适合完整复盘。完整版本保留原始数据身份、校准表、运行日志、环境文件、全部中间产品和发布包，适合研究级复查。

这三种规模应共享同一证据链，而不是变成三套互不相干的材料。小样本可以不包含大型 Measurement Set，但应保留与完整包相同的字段名称；标准包可以不执行完整重校准，但应说明校准表来源；完整包可以包含更多文件，但不应改变核心判断结构。这样课程演示、练习作业和研究复查可以在同一框架内切换。


![文件级示例仓库教学循环](figures/practical_file_teaching_replay_cycle.png)

图 9.32.5 展示了文件级示例仓库的教学循环。同一个仓库可以支撑小样本数据、参数改动、局部重跑、产品比较、失败样本和报告复查，但所有任务都回到同一组清单、参数、产品和限制声明。


这种分层也能把数学和软件连接起来。小样本中可以直接比较 beam 改变如何影响亮温灵敏度；标准包中可以比较不同权重下的源表完整性和可靠性；完整包中可以追踪 flag、校准表和成像参数如何共同改变最终误差预算。每一层都应保留相同的科学问题，而不是把实践练习降格为命令复制。


### 9.32.7 教学案例：低面亮度连续谱复盘发布包

延续第 9.31 节的低面亮度连续谱案例，文件级发布包可以把复盘拆成四个可复查任务。第一，建立 `data_manifest.yaml`，记录归档项目、观测日期、频窗、field、主波束范围、pipeline 产品版本和校验和。第二，建立 `weblog_evidence_board.md`，把短基线 flag、校准解、强源残差和主波束区域映射到科学目标。第三，建立 `imaging_low_surface_brightness.yaml`，固定自然权重、uv taper、多尺度 clean、mask 和主波束裁剪。第四，建立 `limit_statement.md`，说明非探测或上限只适用于某一角尺度、频率和亮温阈值。

这个案例可以包含两个对照运行。第一个运行保留 pipeline 风格的高分辨率权重，用于展示致密源成像为何不能直接回答低面亮度问题。第二个运行使用 taper 和多尺度 clean，用于展示目标尺度下的亮温灵敏度改善。两个运行都应生成产品清单和差异表。差异表不只比较 rms，还要比较共同 beam 后的积分通量、残差结构、强源排除区域、主波束范围和上限计算。

发布包中的报告可以按这样的顺序组织：科学目标，数据身份，质量证据，处理选择，产品比较，验证门，限制声明。这样报告不是简单叙述“做了哪些步骤”，而是说明每一步怎样影响目标量。若短基线不足导致最大可恢复尺度小于目标尺度，报告必须把这一点写进限制声明；若 taper 后残差仍受强源限制，误差预算必须从热噪声转为局部残差主导；若源搜索只用于屏蔽致密源，源表可靠性不应被误写成科学源表可靠性。

这个发布包还可以设计失败样本。把主波束边缘区域故意纳入测量，会得到不稳定上限；把高分辨率图像直接用于低面亮度通量，会低估弥散结构；把旧源表和新图像混用，会造成选择函数不一致。失败样本不进入最终结论，但它们能展示为什么清单、溯源图和验证门不是形式主义。


### 9.32.8 本节结论

文件级示例仓库把射电干涉测量的实践结果从“方法描述”推进到“可复查对象”。一个结果由输入数据、校准状态、处理参数、软件环境和人工判断共同决定。发布包需要让这些对象都有身份，并把图像、源表、测量表、验证报告和限制声明绑定到同一运行编号。

这种做法的教学价值在于，它把理论、软件和科学判断放在同一证据链中。权重、taper、mask、源搜索阈值和误差预算不再只是文字概念，而是可以通过文件、产品差异和验证门被审查。第 9.28 到 9.32 节共同形成一条可复现实践链：项目模板定义结构，最小运行定义事件，归档复盘定义判断，文件级发布包定义复查边界。
